# 04. Addestrare un piccolo decision model gratuitamente

Ora facciamo il fine-tuning di un piccolo modello base open sul JSONL prodotto dal notebook 03. Il codice usa LoRA, che aggiorna un piccolo adapter invece dell'intero modello, quindi gira su hardware modesto.

Installa prima l'extra opzionale con `uv sync --extra train`. Vengono aggiunti solo torch, transformers, peft e accelerate. Non e coinvolto alcun servizio a pagamento.

L'approccio segue lavori liberi come Tev1, minimind-jev e Strands Decider.


![Pipeline di training](../images/training_pipeline.png)

*Il JSONL del notebook 03 addestra un piccolo adapter LoRA sopra un modello base open. L'adapter viene salvato localmente e resta piccolo.*


## Passo 1. Verificare se l'extra di training e disponibile

Proviamo a importare torch. Se manca, la cella stampa il comando esatto per installarlo invece di sollevare un errore. Questo mantiene il notebook leggibile prima dell'installazione.


In [ ]:
try:
    import torch
    print('torch', torch.__version__)
except ImportError:
    print('run: uv sync --extra train')


## Passo 2. Caricare i record convertiti

Il notebook 03 deve essere stato eseguito prima. Leggiamo il JSONL e contiamo i record per sapere quanti esempi di training abbiamo.


In [ ]:
from decision_qa import read_jsonl
records = read_jsonl('data/pubmedqa_decisions.jsonl')
print(len(records))


In [ ]:
assert len(records) > 0
print('dataset loaded')


## Passo 3. Costruire coppie testo-risposta

L'helper di training seleziona la domanda Choice di ogni record e la trasforma in un prompt piu una risposta di un token. Esaminiamo il risultato prima di addestrare.


In [ ]:
from decision_qa.train import build_text_pairs
pairs = build_text_pairs(records)
print(len(pairs))


In [ ]:
prompt, answer = pairs[0]
print(prompt[:400])


In [ ]:
print(repr(answer))
assert answer.strip() in 'ABCDEFGHIJ'


## Passo 4. Perche mascheriamo il prompt

Durante il training calcoliamo la loss solo sul token di risposta. I token del prompt hanno etichetta -100 cosi il modello non viene premiato per memorizzare l'input. Questo e il fine-tuning standard di un modello causale e mantiene l'esempio piccolo.


## Passo 5. Impostazioni di training

Il modello base e open e piccolo. LoRA aggiunge un adapter di rango 16 sulle proiezioni dell'attenzione. Un'epoca basta per una dimostrazione; un vero esperimento di tesi varierebbe learning rate ed epoche su uno split di validazione.


In [ ]:
import os
base_model = os.getenv('DM_BASE_MODEL', 'Qwen/Qwen2.5-0.5B')
print(base_model)


In [ ]:
assert 'Qwen' in base_model
print('open base model selected')


## Passo 6. Eseguire il training, quando si e pronti

Il flag e spento di default cosi il notebook e sicuro su qualunque macchina. Attivatelo dopo aver installato l'extra. L'adapter viene salvato localmente in `checkpoints/`, che e gitignorato.


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from decision_qa.train import train
    adapter = train('data/pubmedqa_decisions.jsonl', 'checkpoints/dm-lora', epochs=1)
    print('saved', adapter)
else:
    print('training skipped on purpose')


## Passo 7. Cosa cambia in un decision model completo

Questa versione LoRA scrive un solo token di risposta. Un decision model completo, come Tev1 o Strands Decider, sostituisce la testa linguistica con uno scorer sulle opzioni dichiarate. Entrambi producono una distribuzione; lo scorer lo fa in un solo passaggio senza decodifica.

Dopo il training, rieseguite il notebook 05 per confrontare l'adapter con il modello open locale e con la route ospitata gratuita. Mantenete tutto gratuito.
